# VAE JAX CelebA Kaggle TPU v5e-8 Pipeline (SiT-B + moe1)


In [ ]:
%cd /kaggle/working
!rm -rf RAE
!git clone https://github.com/sontungkieu/RAE
%cd /kaggle/working/RAE
!git checkout jax-vae-sit-moe1
!curl -LsSf https://astral.sh/uv/install.sh | sh
!ln -sf /root/.local/bin/uv /usr/local/bin/uv


In [ ]:
import os

os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/.venv"
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"

!uv sync -q
!uv run python scripts/clear_elf_execstack.py --package jaxlib
print("Synced the repo dependencies into /tmp/.venv. The package-backed steps below all go through uv run.")


In [ ]:
import os
import pathlib

os.environ["JAX_PLATFORMS"] = "tpu,cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["PYOPENGL_PLATFORM"] = "egl"

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    wandb_token = secrets.get_secret("WANDB2")
    hf_token = secrets.get_secret("HF_TOK_WRITE_KAGGLE")

    os.environ["WANDB_API_KEY"] = wandb_token
    os.environ["WANDB_KEY"] = wandb_token
    os.environ["HF_TOKEN"] = hf_token

    netrc = pathlib.Path.home() / ".netrc"
    netrc.write_text(f"machine api.wandb.ai login user password {wandb_token}\n")
    os.chmod(netrc, 0o600)
    print("Loaded Kaggle secrets for wandb and Hugging Face.")
except Exception as exc:
    print(f"Skipping Kaggle secret bootstrap: {exc}")


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged

export JAX_PLATFORMS="tpu,cpu"
export XLA_PYTHON_CLIENT_PREALLOCATE="false"

uv run python - <<'PY'
import os
import sys

import jax
import jaxlib

print("python executable:", sys.executable)
print("jax version:", jax.__version__)
print("jaxlib version:", jaxlib.__version__)
print("host cpu cores:", os.cpu_count())
print("default backend:", jax.default_backend())
print("local device count:", jax.local_device_count())
print("devices:", jax.devices())
PY


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

echo "Using backend-native StabilityVAE; no external RAE decoder download is required."


In [ ]:
from pathlib import Path

repo_root = Path("/kaggle/working/RAE")
celeba_src_dir = Path("/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba")
celeba_split_csv = Path("/kaggle/input/datasets/jessicali9530/celeba-dataset/list_eval_partition.csv")
celeba_root = Path("/kaggle/working/celeba256_imgfolder")
stage1_cfg_path = repo_root / "configs" / "stage1" / "pretrained" / "CelebA256_StabilityVAE_jax_tpuv5e8.yaml"
stage2_cfg_path = repo_root / "configs" / "stage2" / "training" / "CelebA256_SiT-B_StabilityVAE_moe1_jax_tpuv5e8.yaml"
source_gmm_path = Path("/kaggle/working/celeba256_source_gmm.npz")
fid_stats_path = Path("/kaggle/working/celeba256_val_fid_stats_cpu.pkl")
stage1_single_recon_path = Path("/kaggle/working/celeba256_vae_stage1_single_recon_tpu.png")
stage1_recon_dir = Path("/kaggle/working/celeba256_vae_stage1_recon_val_tpu")
stage2_results_dir = Path("/kaggle/working/results_jax_tpu")

image_size = 256
recon_batch_size = 16
recon_num_workers = 16
fid_num_workers = 32
recon_limit = 2048  # bỏ limit nếu muốn reconstruct toàn bộ val split

print("repo_root:", repo_root)
print("celeba_root:", celeba_root)
print("stage1_cfg_path:", stage1_cfg_path)
print("stage2_cfg_path:", stage2_cfg_path)
print("source_gmm_path:", source_gmm_path)
print("fid_stats_path:", fid_stats_path)
print("stage2_results_dir:", stage2_results_dir)


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PY'
from pathlib import Path
celeba_src_dir = Path("/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba")
celeba_split_csv = Path("/kaggle/input/datasets/jessicali9530/celeba-dataset/list_eval_partition.csv")
celeba_root = Path("/kaggle/working/celeba256_imgfolder")
image_size = 256

import concurrent.futures as futures
import os

import numpy as np
import pandas as pd
from PIL import Image


def center_crop_arr(pil_image: Image.Image, target_size: int) -> Image.Image:
    while min(*pil_image.size) >= 2 * target_size:
        pil_image = pil_image.resize(tuple(x // 2 for x in pil_image.size), resample=Image.BOX)

    scale = target_size / min(*pil_image.size)
    pil_image = pil_image.resize(tuple(round(x * scale) for x in pil_image.size), resample=Image.BICUBIC)
    arr = np.asarray(pil_image)
    crop_y = (arr.shape[0] - target_size) // 2
    crop_x = (arr.shape[1] - target_size) // 2
    return Image.fromarray(arr[crop_y : crop_y + target_size, crop_x : crop_x + target_size])


def prepare_row(row):
    split = {0: "train", 1: "val", 2: "test"}[int(row.partition)]
    src = celeba_src_dir / row.image_id
    dst = celeba_root / split / "face" / row.image_id
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return None
    with Image.open(src) as image:
        cropped = center_crop_arr(image.convert("RGB"), image_size)
        cropped.save(dst, quality=95)
    return dst


df = pd.read_csv(celeba_split_csv)
rows = list(df.itertuples(index=False))
existing = sum(1 for _ in celeba_root.rglob("*.jpg"))

if existing < len(rows):
    max_workers = min(16, os.cpu_count() or 8)
    with futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        created = sum(1 for item in executor.map(prepare_row, rows, chunksize=128) if item is not None)
    print(f"Created {created} resized CelebA images under {celeba_root}")
else:
    print(f"CelebA imagefolder already prepared under {celeba_root}")

print("train images:", sum(1 for _ in (celeba_root / "train").rglob("*.jpg")))
print("val images:", sum(1 for _ in (celeba_root / "val").rglob("*.jpg")))
print("test images:", sum(1 for _ in (celeba_root / "test").rglob("*.jpg")))
PY


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PYCFG'
from pathlib import Path
import textwrap

repo_root = Path("/kaggle/working/RAE")
celeba_root = Path("/kaggle/working/celeba256_imgfolder")
stage1_cfg_path = repo_root / "configs" / "stage1" / "pretrained" / "CelebA256_StabilityVAE_jax_tpuv5e8.yaml"
stage2_cfg_path = repo_root / "configs" / "stage2" / "training" / "CelebA256_SiT-B_StabilityVAE_moe1_jax_tpuv5e8.yaml"
source_gmm_path = Path("/kaggle/working/celeba256_source_gmm.npz")
fid_stats_path = Path("/kaggle/working/celeba256_val_fid_stats_cpu.pkl")

stage1_cfg_text = textwrap.dedent(
    """
    stage_1:
      target: stage1.StabilityVAE
      params:
        sample_size: 256
        latent_channels: 4
        downsample_factor: 8
        raw_mean: [0.865, -0.278, 0.216, 0.374]
        raw_std: [4.86, 5.32, 3.94, 3.99]
        final_mean: 0.0
        final_std: 0.5
    """
).strip() + "\n"

stage2_cfg_text = textwrap.dedent(
    f"""
    stage_1:
      target: stage1.StabilityVAE
      ckpt: null
      params:
        sample_size: 256
        latent_channels: 4
        downsample_factor: 8
        raw_mean: [0.865, -0.278, 0.216, 0.374]
        raw_std: [4.86, 5.32, 3.94, 3.99]
        final_mean: 0.0
        final_std: 0.5

    stage_2:
      target: stage2.models.SiT.SiT
      ckpt: null
      params:
        input_size: 32
        patch_size: 2
        in_channels: 4
        hidden_size: 768
        depth: 12
        num_heads: 12
        mlp_ratio: 4.0
        class_dropout_prob: 0.0
        num_classes: 1
        use_qknorm: false
        use_swiglu: true
        use_rope: true
        use_rmsnorm: true
        wo_shift: false

    transport:
      params:
        path_type: 'Linear'
        prediction: 'velocity'
        loss_weight: null
        time_dist_type: 'uniform'

    sampler:
      mode: ODE
      params:
        sampling_method: 'euler'
        num_steps: 50
        atol: 1.0e-6
        rtol: 1.0e-3
        reverse: false

    guidance:
      method: 'cfg'
      scale: 1.0
      t_min: 0.0
      t_max: 1.0

    misc:
      latent_size: [4, 32, 32]
      num_classes: 1
      time_dist_shift_dim: 4096
      time_dist_shift_base: 4096

    source:
      enabled: true
      kind: gmm_moe1
      gmm_stats_path: '{source_gmm_path.as_posix()}'
      num_modes: 4
      condition_dim: 16
      hidden_channels: 64
      router_temperature: 2.0
      soft_moe: true
      balance_loss_weight: 0.1
      entropy_loss_weight: 1.0e-2
      var_kl_loss_weight: 1.0
      target_variance: 1.0
      logvar_min: -8.0
      logvar_max: 4.0
      var_floor: 1.0e-5
      posterior_eps: 1.0e-6
      weight_prior: 1.0e-2
      em_iters: 100
      em_tol: 1.0e-4
      em_restarts: 3
      dead_count_threshold: 1.0
      active_mode_fraction_threshold: 0.01

    eval:
      data_path: '{(celeba_root / "val").as_posix()}'
      eval_every: 5000
      batch_size: 4
      num_workers: 16
      prefetch_factor: 4
      random_flip: false
      max_batches: 32
      eval_model: false
      fid_ref: '{fid_stats_path.as_posix()}'
      fid_every: 5000
      fid_num_samples: 4096
      fid_per_proc_batch_size: 4
      fid_batch_size: 128

    training:
      global_seed: 0
      epochs: 200
      global_batch_size: 128
      grad_accum_steps: 1
      ema_decay: 0.9995
      num_workers: 16
      prefetch_factor: 4
      random_flip: true
      log_rae_latent_stats: true
      log_activation_stats: true
      log_every: 10
      ckpt_every: 210000
      sample_every: 5000
      base_lr: 0.0001
      final_lr: 0.00001
      beta: [0.9, 0.95]
      wd: 0.0
      schedule_type: 'linear'
      decay_start_epoch: 150
      decay_end_epoch: 200
      clip_grad: 1.0
    """
).strip() + "\n"

stage1_cfg_path.parent.mkdir(parents=True, exist_ok=True)
stage2_cfg_path.parent.mkdir(parents=True, exist_ok=True)
stage1_cfg_path.write_text(stage1_cfg_text, encoding="utf-8")
stage2_cfg_path.write_text(stage2_cfg_text, encoding="utf-8")

print(f"Wrote {stage1_cfg_path}")
print(f"Wrote {stage2_cfg_path}")
print(f"Stage 2 source artifact path: {source_gmm_path}")
PYCFG


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PYVIEW'
from pathlib import Path

for path in [
    Path("/kaggle/working/RAE/configs/stage1/pretrained/CelebA256_StabilityVAE_jax_tpuv5e8.yaml"),
    Path("/kaggle/working/RAE/configs/stage2/training/CelebA256_SiT-B_StabilityVAE_moe1_jax_tpuv5e8.yaml"),
]:
    print(f"=== {path} ===")
    print(path.read_text())
PYVIEW


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PYCHECK'
from pathlib import Path

sample_image_path = next(path for path in (Path("/kaggle/working/celeba256_imgfolder") / "val" / "face").iterdir() if path.is_file())
print("sample image:", sample_image_path)
print("The StabilityVAE + moe1 flow does not need a bootstrap latent-stat pass.")
print("Build the source GMM artifact before Stage 2 training or resume.")
PYCHECK


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged

sample_image=$(find /kaggle/working/celeba256_imgfolder/val/face -type f -print -quit)
[ -n "${sample_image}" ]

uv run python src_jax/stage1_sample.py \
  --config configs/stage1/pretrained/CelebA256_StabilityVAE_jax_tpuv5e8.yaml \
  --image "${sample_image}" \
  --output /kaggle/working/celeba256_vae_stage1_single_recon_tpu.png


In [ ]:
from IPython.display import Image, display
display(Image(filename=stage1_single_recon_path))

In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged

uv run python src_jax/reconstruct_folder.py \
  --config configs/stage1/pretrained/CelebA256_StabilityVAE_jax_tpuv5e8.yaml \
  --input /kaggle/working/celeba256_imgfolder/val \
  --output-dir /kaggle/working/celeba256_vae_stage1_recon_val_tpu \
  --batch-size 16 \
  --num-workers 16 \
  --limit 2048


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/build_fid_stats.py \
  --input /kaggle/working/celeba256_imgfolder/val \
  --output /kaggle/working/celeba256_val_fid_stats_cpu.pkl \
  --image-size 256 \
  --batch-size 64 \
  --num-workers 8


## Optional: launch Stage 2 JAX VAE + SiT-B training on Kaggle TPU v5e-8


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged

uv run python src_jax/build_source_gmm.py \
  --config configs/stage1/pretrained/CelebA256_StabilityVAE_jax_tpuv5e8.yaml \
  --input /kaggle/working/celeba256_imgfolder/train \
  --output /kaggle/working/celeba256_source_gmm.npz \
  --batch-size 16 \
  --num-workers 16 \
  --precision bf16 \
  --num-modes 4 \
  --em-iters 100 \
  --em-restarts 3


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged

timestamp="$(TZ=Asia/Bangkok date +%Y%m%d-%H%M%S)"
run_name="CelebA256_SiT-B_StabilityVAE_moe1_jax_tpuv5e8-${timestamp}"

export ENTITY="TungBangDSLab"
export PROJECT="vae-jax-celeba256-tpuv5e8-sitb-moe1"

uv run python src_jax/train.py \
  --config configs/stage2/training/CelebA256_SiT-B_StabilityVAE_moe1_jax_tpuv5e8.yaml \
  --data-path /kaggle/working/celeba256_imgfolder \
  --results-dir /kaggle/working/results_jax_tpu \
  --precision bf16 \
  --exp-name "${run_name}" \
  --wandb \
  --wandb-entity TungBangDSLab \
  --wandb-project "${PROJECT}" \
  --set training.global_batch_size=64 \
  --set training.num_workers=16 \
  --set training.prefetch_factor=4 \
  --set training.ckpt_every=210000 \
  --set training.log_rae_latent_stats=true \
  --set training.log_activation_stats=true \
  --set eval.num_workers=16 \
  --set eval.prefetch_factor=4 \
  --set eval.fid_ref=/kaggle/working/celeba256_val_fid_stats_cpu.pkl \
  --set eval.fid_every=10000 \
  --set eval.fid_num_samples=4096
